In [1]:
from pathlib import Path 
import math
import pickle
import bisect
import numpy as np
import time

In [2]:
DEBUG = True
W_SIZE = 1000
AREA_MAX = W_SIZE * W_SIZE
AREA_TH = 0.996
INF = 10 ** 18

def debug_print(*args, **kwargs):
    if DEBUG:
        print(*args, **kwargs)

In [3]:
def fastcopy(obj):
    return pickle.loads(pickle.dumps(obj, -1))

class Env:
    def __init__(self, input_txt_path: Path):
        self.W, self.D, self.N, self.a = self._input(input_txt_path)

    def _input(self, txt_path):
        with open(txt_path, mode="r") as file:
            lines = file.readlines()
        W, D, N = map(int, lines[0].split())
        a = []
        for line, d in zip(lines[1:], range(D)):
            d = list(map(int, line.split()))
            a.append(d)
        return W, D, N, a

class Area:
    def __init__(self, id_val: int, self_area: int, target_area: int):
        self.id = id_val
        self.self_area = self_area
        self.target_area = target_area
        self.need_first = False

In [4]:
def check_ans(ans: list[tuple[int]]):
    for d in range(len(ans)):
        for coordinates in ans[d]:
            x1, y1, x2, y2 = coordinates
            if x1 < 0 or x2 > W_SIZE or y1 < 0 or y2 > W_SIZE:
                return False
    return True

# def check_ans(ans:list[list[int]], env:Env):
#     for d in range(len(ans)):
#         for coordinates in ans[d]:
#             x1, y1, x2, y2 = coordinates
#             if x1 < 0 or x2 > W_SIZE or y1 < 0 or y2 > W_SIZE:
#                 return False
#     for d in range(len(ans)):
#         for ans_vec, true_area in zip(ans[d], env.a[d]):
#             ans_area = (ans_vec[2] - ans_vec[0]) * (ans_vec[3] - ans_vec[1])
#             if ans_area < true_area:
#                 return False
#     return True

In [5]:
def get_diff_max(target_areas: list[list[int]], env: Env):
    now_target_areas = fastcopy(target_areas)
    target_dw = len(target_areas)

    get_ind_from_area = [dict() for _ in range(target_dw)]
    for d in range(target_dw):
        tmp_dict = dict()
        for tn in range(env.N):
            val = now_target_areas[d][tn]
            tmp_dict[val] = tn
        get_ind_from_area[d] = tmp_dict

    diff_max = []
    for n in range(env.N):
        tmp_diff_max = []
        len_n = len(now_target_areas[0])
        for d in range(target_dw):
            for tn in range(len_n):
                left_area = now_target_areas[d][tn]
                seach_areas = []
                for next_d in range(target_dw):
                    if d == next_d:
                        seach_areas.append(left_area)
                        continue
                    b_ind = bisect.bisect_left(now_target_areas[next_d], left_area)
                    if b_ind == len_n:
                        break
                    seach_areas.append(now_target_areas[next_d][b_ind])
                if len(seach_areas) != target_dw:
                    continue
                diff = max(seach_areas) - min(seach_areas)
                max_area = max(seach_areas)
                inds = []
                for si, area in enumerate(seach_areas):
                    inds.append(get_ind_from_area[si][area])
                tmp_diff_max.append((diff, max_area, inds, seach_areas))
        sorted_diff_max = sorted(tmp_diff_max)
        add_diff, add_max_area, add_inds, del_areas = sorted_diff_max[0]
        diff_max.append((add_diff, add_max_area, add_inds))
        for si, area in enumerate(del_areas):
            now_target_areas[si].remove(area)

    return diff_max

In [6]:
def calc_cost(ans: list[list[int]], env: Env):
    partial_cost = 0
    area_cost = 0

    hs = set()
    vs = set()
    for d in range(env.D):
        hs2 = set()
        vs2 = set()
        for k in range(env.N):
            i0, j0, i1, j1 = ans[d][k]
            area = (i1 - i0) * (j1 - j0)
            if env.a[d][k] > area:
                area_cost += 100 * (env.a[d][k] - area)
            for j in range(j0, j1):
                if i0 > 0:
                    hs2.add((i0, j))
                if i1 < env.W:
                    hs2.add((i1, j))
            for i in range(i0, i1):
                if j0 > 0:
                    vs2.add((j0, i))
                if j1 < env.W:
                    vs2.add((j1, i))

        if d > 0:
            partial_cost += len(hs ^ hs2)
            partial_cost += len(vs ^ vs2)

        hs = hs2
        vs = vs2
    return partial_cost + area_cost + 1

In [7]:
def dicision_pos(ans, n, env: Env, one_day_area, used_area, now_lr, now_ud, x1, y1, x2, y2, how="aspect"):
    area_diffs = []
    for i, area in enumerate(one_day_area):
        if used_area[i]:
            continue
        len1 = math.ceil(area / now_lr)
        len2 = math.ceil(area / now_ud)
        if len1 * now_lr < len2 * now_ud:
            direct = "ud"
            if how == "diff":
                score = len1 * now_lr - area
            elif how == "aspect":
                score = 1 - min(len1, now_lr) / max(len1, now_lr)
            area_diffs.append((score, len1, direct, i))
        else:
            direct = "lr"
            if how == "diff":
                score = len2 * now_ud - area
            elif how == "aspect":
                score = 1 - min(len2, now_ud) / max(len2, now_ud)
            area_diffs.append((score, len2, direct, i))
    _, min_len, min_direct, min_i = min(area_diffs)
    if n == env.N - 1:
        ans[min_i] = (x1, y1, x2, y2)
        assigin_area = (x2 - x1) * (y2 - y1)
    elif min_direct == "ud":
        ans[min_i] = (x1, y1, x2, y1 + min_len)
        assigin_area = (x2 - x1) * min_len
        y1 += min_len
    elif min_direct == "lr":
        ans[min_i] = (x2 - min_len, y1, x2, y2)
        assigin_area = min_len * (y2 - y1)
        x2 -= min_len
    else:
        raise ValueError("invalid direct")
    used_area[min_i] = True

    return x1, y1, x2, y2, assigin_area - one_day_area[min_i]

In [8]:
def one_day_greedy_solve(one_day_area: list[int], env: Env):
    x1 = 0
    y1 = 0
    x2 = W_SIZE
    y2 = W_SIZE
    remain_area = W_SIZE * W_SIZE - sum(one_day_area)

    ans = [None for _ in range(env.N)]
    used_area = [False] * env.N
    for n in range(env.N):
        remain_n = env.N - n
        now_lr = x2 - x1
        now_ud = y2 - y1
        if remain_n * (W_SIZE - 1) <= remain_area:
            # 残り面積が広いときはアスペクト比を貪欲探索
            x1, y1, x2, y2, assigin_area = dicision_pos(ans, n, env, one_day_area, used_area, now_lr, now_ud, x1, y1, x2, y2, how="aspect")
        else:
            # 残り面積が狭いときは面積を有効活用
            x1, y1, x2, y2, assigin_area = dicision_pos(ans, n, env, one_day_area, used_area, now_lr, now_ud, x1, y1, x2, y2, how="diff")
        remain_area -= assigin_area
    
    return ans

In [9]:
def common_area_solver_core_opt(ret_area_objs, env: Env):

    ans = []
    target_dw = len(ret_area_objs)

    for d in range(target_dw):
        is_need_areas = [area for area in ret_area_objs[d] if area.need_first]
        is_not_need_areas = [area for area in ret_area_objs[d] if not area.need_first]

        x1 = 0
        y1 = 0
        x2 = W_SIZE
        y2 = W_SIZE

        now_ans = [None for _ in range(env.N)]
        for n in range(env.N):
            area_diffs = []
            now_lr = x2 - x1
            now_ud = y2 - y1
            if is_need_areas:
                for target_area in is_need_areas:
                    len1 = math.ceil(target_area.target_area / now_lr)
                    len2 = math.ceil(target_area.target_area / now_ud)
                    if len1 * now_lr < len2 * now_ud:
                        direct = "ud"
                        diff = len1 * now_lr - target_area.target_area
                        area_diffs.append((diff, len1, direct, target_area.id))
                    else:
                        direct = "lr"
                        diff = len2 * now_ud - target_area.target_area
                        area_diffs.append((diff, len2, direct, target_area.id))
            elif is_not_need_areas:
                for target_area in is_not_need_areas:
                    len1 = math.ceil(target_area.target_area / now_lr)
                    len2 = math.ceil(target_area.target_area / now_ud)
                    if len1 * now_lr < len2 * now_ud:
                        direct = "ud"
                        diff = len1 * now_lr - target_area.target_area
                        area_diffs.append((diff, len1, direct, target_area.id))
                    else:
                        direct = "lr"
                        diff = len2 * now_ud - target_area.target_area
                        area_diffs.append((diff, len2, direct, target_area.id))
            else:
                raise ValueError("target is empty")

            min_diff, min_len, min_direct, area_id = sorted(area_diffs)[0]
            if n == env.N - 1:
                now_ans[area_id] = (x1, y1, x2, y2)
                break
            if min_direct == "ud":
                now_ans[area_id] = (x1, y1, x2, y1 + min_len)
                y1 += min_len
            else:
                now_ans[area_id] = (x2 - min_len, y1, x2, y2)
                x2 -= min_len

            if is_need_areas:
                is_need_areas = [area for area in is_need_areas if area.id != area_id]
            elif is_not_need_areas:
                is_not_need_areas = [area for area in is_not_need_areas if area.id != area_id]
        ans.append(now_ans)

    if not check_ans(ans):
        raise ValueError("failed check ans")

    return ans

In [10]:
def common_area_solver_core_aspect(ret_area_objs, env: Env):

    ans = []
    target_dw = len(ret_area_objs)

    for d in range(target_dw):
        is_need_areas = [area for area in ret_area_objs[d] if area.need_first]
        is_not_need_areas = [area for area in ret_area_objs[d] if not area.need_first]

        x1 = 0
        y1 = 0
        x2 = W_SIZE
        y2 = W_SIZE

        now_ans = [None for _ in range(env.N)]
        for n in range(env.N):
            area_diffs = []
            now_lr = x2 - x1
            now_ud = y2 - y1
            if is_need_areas:
                for target_area in is_need_areas:
                    len1 = math.ceil(target_area.target_area / now_lr)
                    len2 = math.ceil(target_area.target_area / now_ud)
                    if len1 * now_lr < len2 * now_ud:
                        direct = "ud"
                        aspect = 1 - min(len1, now_lr) / max(len1, now_lr)
                        diff = len1 * now_lr - target_area.target_area
                        area_diffs.append((aspect, diff, len1, direct, target_area.id))
                    else:
                        direct = "lr"
                        aspect = 1 - min(len2, now_ud) / max(len2, now_ud)
                        diff = len2 * now_ud - target_area.target_area
                        area_diffs.append((aspect, diff, len2, direct, target_area.id))
            elif is_not_need_areas:
                for target_area in is_not_need_areas:
                    len1 = math.ceil(target_area.target_area / now_lr)
                    len2 = math.ceil(target_area.target_area / now_ud)
                    if len1 * now_lr < len2 * now_ud:
                        direct = "ud"
                        aspect = 1 - min(len1, now_lr) / max(len1, now_lr)
                        diff = len1 * now_lr - target_area.target_area
                        area_diffs.append((aspect, diff, len1, direct, target_area.id))
                    else:
                        direct = "lr"
                        aspect = 1 - min(len2, now_ud) / max(len2, now_ud)
                        diff = len2 * now_ud - target_area.target_area
                        area_diffs.append((aspect, diff, len2, direct, target_area.id))
            else:
                raise ValueError("target is empty")

            _, _, min_len, min_direct, area_id = sorted(area_diffs)[0]
            if n == env.N - 1:
                now_ans[area_id] = (x1, y1, x2, y2)
                break
            if min_direct == "ud":
                now_ans[area_id] = (x1, y1, x2, y1 + min_len)
                y1 += min_len
            else:
                now_ans[area_id] = (x2 - min_len, y1, x2, y2)
                x2 -= min_len

            if is_need_areas:
                is_need_areas = [area for area in is_need_areas if area.id != area_id]
            elif is_not_need_areas:
                is_not_need_areas = [area for area in is_not_need_areas if area.id != area_id]
        ans.append(now_ans)

    if not check_ans(ans):
        raise ValueError("failed check ans")

    return ans

In [11]:
def common_area_solver(target_areas: list[list[int]], env: Env):

    ans = []

    target_dw = len(target_areas)
    diff_max = get_diff_max(target_areas, env)

    # エリアオブジェクトを作成
    area_objs = []
    for target_area in target_areas:
        area_obj = []
        for i in range(env.N):
            area_obj.append(Area(id_val=i, self_area=target_area[i], target_area=target_area[i]))
        area_objs.append(area_obj)

    # 目標面積を決定
    ret_area_objs = fastcopy(area_objs)
    for _, max_area, pattern in diff_max:
        now_area_objs = fastcopy(ret_area_objs)
        for d in range(target_dw):
            now_area_objs[d][pattern[d]].target_area = max_area
            now_area_objs[d][pattern[d]].need_first = True
        is_ok = True
        for d in range(target_dw):
            sum_area = sum(now_area_objs[d][i].target_area for i in range(env.N))
            if sum_area / AREA_MAX > AREA_TH:
                is_ok = False
                break
        if is_ok:
            ret_area_objs = fastcopy(now_area_objs)
        else:
            break

    try:
        ans = common_area_solver_core_aspect(ret_area_objs, env)
        return ans
    except Exception as e:
        debug_print(f"Err: common_area_solver_core_aspect {e}")
        ans = common_area_solver_core_opt(ret_area_objs, env)
        return ans

In [12]:
def fin_solver(target_areas: list[list[int]], env: Env):
    ret = []
    if len(target_areas) == 1:
        ret = [one_day_greedy_solve(target_areas[0], env)]
    else:
        try:
            ret = common_area_solver(target_areas, env)
        except Exception as e:
            debug_print(f"Err: common_area_solver {e}")
            for d in range(len(target_areas)):
                ret.append(one_day_greedy_solve(env.a[d], env))
    return ret

In [13]:
def solve(env: Env):
    best_cost = INF
    best_ans = None
    best_shift = None

    start_time = time.time()
    shift_list = list(set([1, 2] + [int(x) for x in np.linspace(3, env.D, 8)]))
    for shift in shift_list:
        ans = []
        for d in range(0, env.D, shift):
            ans += fin_solver(env.a[d:d+shift], env)
        cost = calc_cost(ans, env)

        if cost < best_cost:
            best_cost = cost
            best_ans = ans
            best_shift = shift
        if time.time() - start_time > 2.4:
            debug_print("over!!!")
            break

    debug_print(f"best:{best_cost} shift:{best_shift}/{env.D}")
    return best_ans

In [14]:
def main():
    for i in range(100):
        debug_print(f"----- {i}/100 -----")
        input_txt_path = Path(f"./in/{str(i).zfill(4)}.txt")
        output_txt_path = Path(f"./out/{str(i).zfill(4)}.txt")
        env = Env(input_txt_path)

        ans = solve(env)

        str_ans = []
        for d in range(env.D):
            for k in range(env.N):
                i0, j0, i1, j1 = ans[d][k]
                str_ans.append(f"{i0} {j0} {i1} {j1}")

        with open(output_txt_path, mode="w") as file:
            file.write("\n".join(str_ans))
main()

----- 0/100 -----
best:9619 shift:3/5
----- 1/100 -----
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect division by zero
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect failed check ans
Err: common_area_solver_core_aspect divisio

In [15]:
# import time

# start_time = time.time()

# i = 94
# input_txt_path = Path(f"./in/{str(i).zfill(4)}.txt")
# output_txt_path = Path(f"./out/{str(i).zfill(4)}.txt")

# env = Env(input_txt_path)
# ans = solve(env)

# str_ans = []
# for d in range(env.D):
#     for k in range(env.N):
#         i0, j0, i1, j1 = ans[d][k]
#         str_ans.append(f"{i0} {j0} {i1} {j1}")

# with open(output_txt_path, mode="w") as file:
#     file.write("\n".join(str_ans))

# debug_print(time.time() - start_time)